# Merged BERTopic - Build and Save

Purpose: this notebook only merges the outlet BERTopic models, assigns the merged model back to articles, and saves the canonical merged artefacts for downstream analysis.

Thesis note: the finalized merged run used for thesis analysis is the frozen snapshot `2026-04-03_merged_v1`. This notebook documents the build procedure and can reproduce the canonical merged outputs, but the frozen snapshot remains the reference run if a later rerun drifts.

Do analysis, pivots, topic labeling, and visualizations in `Merged_BERTopic_Analysis.ipynb`.

## 1. Setup

In [ ]:
from pathlib import Path
import shutil
import sys

import pandas as pd
from bertopic import BERTopic

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "Thesis":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODULE_DIR = PROJECT_ROOT / "02_TopicModeling"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import merged_outlets_analysis as moa
from merged_outlets_analysis import OUTLET_SPECS

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

MODEL_BASE_DIRS = [PROJECT_ROOT / "02_TopicModeling" / "outputs"]
MERGE_ORDER = [
    "tagesschau",
    "rt",
    "antispiegel",
    "tichys",
    "nius",
    "compact",
    "deutschlandkurier",
]
MIN_SIMILARITY = 0.7
RERUN_FROZEN_LABEL = "final_merged_topics_thesis_final_rerun"
MERGED_MODEL_DIR = PROJECT_ROOT / moa.DEFAULT_MERGED_MODEL_RELATIVE_PATH
MERGED_TOPIC_TOP30_PATH = PROJECT_ROOT / moa.DEFAULT_MERGED_TOPIC_INFO_TOP30_RELATIVE_PATH
CONSISTENT_TOPIC_LABELS_PATH = PROJECT_ROOT / moa.DEFAULT_SORTED_TOPIC_LABELS_RELATIVE_PATH


## 2. Load outlet models and confirm the source models

In [ ]:
loaded_model_paths = moa.resolve_model_paths(MODEL_BASE_DIRS)
loaded_models = {
    key: BERTopic.load(path, embedding_model=moa.EMBEDDING_MODEL_NAME)
    for key, path in loaded_model_paths.items()
}

raw_counts = pd.read_csv(PROJECT_ROOT / "01_EDAperOutlet" / "df_combined.csv")["source"].value_counts().to_dict()
source_name_map = {
    "tagesschau": "Tagesschau",
    "rt": "RT_de",
    "antispiegel": "Antispiegel",
    "tichys": "Tichys_Einblick",
    "nius": "Nius",
    "compact": "Compact",
    "deutschlandkurier": "Deutschlandkurier",
}

rows = []
for key in MERGE_ORDER:
    model = loaded_models[key]
    topic_info = model.get_topic_info().copy()
    docs = len(model.topics_)
    outliers = int((pd.Series(model.topics_) == -1).sum())
    topics = int(topic_info.loc[topic_info["Topic"] != -1, "Topic"].nunique())
    assigned = docs - outliers
    rows.append({
        "Outlet": OUTLET_SPECS[key].label,
        "RawDocs": raw_counts[source_name_map[key]],
        "ModelDocs": docs,
        "Topics": topics,
        "OutlierPct": round(outliers / docs * 100, 2),
        "DocsPerTopic": round(assigned / topics, 2) if topics else float("nan"),
        "ModelPath": str(loaded_model_paths[key]),
    })

step0_summary = pd.DataFrame(rows)
display(step0_summary)


## 3. Cumulative merge (Tagesschau as the base outlet)

In [ ]:
merge_rows = []
merged_model = None
merged_topic_info_model = None
merged_topics_overview = None

current_keys = []
for idx, key in enumerate(MERGE_ORDER, start=1):
    current_keys.append(key)
    current_models = [loaded_models[k] for k in current_keys]
    merged_model = BERTopic.merge_models(current_models, min_similarity=MIN_SIMILARITY)
    merged_topic_info_model = merged_model.get_topic_info().copy()
    merged_topics_overview = (
        merged_topic_info_model.loc[merged_topic_info_model["Topic"] != -1, ["Topic", "Count", "Name", "Representation"]]
        .sort_values(["Count", "Topic"], ascending=[False, True])
        .reset_index(drop=True)
    )
    merge_rows.append({
        "Step": idx,
        "IncludedOutlets": " + ".join(OUTLET_SPECS[k].label for k in current_keys),
        "NewlyAddedOutlet": OUTLET_SPECS[key].label,
        "AddedOutletTopics": int(step0_summary.loc[step0_summary["Outlet"] == OUTLET_SPECS[key].label, "Topics"].iloc[0]),
        "MergedTopicsNow": int(merged_topics_overview["Topic"].nunique()),
        "MergedRowsIncludingOutlier": int(len(merged_topic_info_model)),
    })

merge_summary = pd.DataFrame(merge_rows)
display(merge_summary)
display(merged_topics_overview.head(30))
print("Final merged topic count excluding -1:", int(merged_topics_overview["Topic"].nunique()))


## 4. Assign the merged model back to all prepared articles

In [ ]:
prepared_by_outlet = moa.load_all_prepared_documents(PROJECT_ROOT)
combined_prepared = moa.combine_prepared_documents(prepared_by_outlet)

merged_articles_base, merged_topic_info_base, merged_umap_model_base = moa.build_merged_article_frame(
    merged_model,
    combined_prepared,
)
merged_topic_info_top30 = moa.build_merged_topic_top_words(
    merged_articles_base,
    merged_topic_info_base,
    n_words=30,
    min_df=2,
)

print("Prepared documents:", len(combined_prepared))
print("Article-level outliers:", int((merged_articles_base["merged_topic"] == -1).sum()))
display(merged_topic_info_base.head(20))
display(
    merged_topic_info_top30.loc[
        merged_topic_info_top30["Topic"] != -1,
        ["DisplayTopic", "Topic", "DisplayLabel", "Count", "TopWords30Str"],
    ]
    .sort_values(["DisplayTopic", "Topic"], na_position="last")
    .head(20)
)


## 5. Save the canonical merged artefacts

In [ ]:
if MERGED_MODEL_DIR.exists():
    shutil.rmtree(MERGED_MODEL_DIR)

merged_model.save(
    MERGED_MODEL_DIR,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=moa.EMBEDDING_MODEL_NAME,
)

articles_path, topic_info_path, metadata_path = moa.export_merged_analysis_cache(
    PROJECT_ROOT,
    merged_articles_base,
    merged_topic_info_base,
    merged_model_path=MERGED_MODEL_DIR,
    metadata_overrides={
        "min_similarity": MIN_SIMILARITY,
        "merge_order": MERGE_ORDER,
        "run_label": RERUN_FROZEN_LABEL,
    },
)
merged_topic_info_top30, merged_topic_top30_path = moa.export_merged_topic_top_words(
    PROJECT_ROOT,
    merged_articles_base,
    merged_topic_info_base,
    output_path=MERGED_TOPIC_TOP30_PATH,
    n_words=30,
    min_df=2,
)
consistent_topic_labels, consistent_topic_labels_path = moa.export_consistent_topic_label_table(
    PROJECT_ROOT,
    merged_topic_info_top30,
    output_path=CONSISTENT_TOPIC_LABELS_PATH,
)

df_combined_topic_path = moa.export_df_combined_with_topic(
    PROJECT_ROOT,
    merged_articles_base,
)

merge_summary_path = PROJECT_ROOT / "data" / "processed" / "merged_step_summary.csv"
merge_summary_path.parent.mkdir(parents=True, exist_ok=True)
merge_summary.to_csv(merge_summary_path, index=False)

merge_overview_path = PROJECT_ROOT / "data" / "processed" / "merged_topics_overview_build.csv"
merged_topics_overview.to_csv(merge_overview_path, index=False)

snapshot_dir = moa.freeze_merged_run_snapshot(
    PROJECT_ROOT,
    run_label=RERUN_FROZEN_LABEL,
    overwrite=True,
)

print("Saved merged model to:", MERGED_MODEL_DIR)
print("Saved merged articles cache to:", articles_path)
print("Saved merged topic info cache to:", topic_info_path)
print("Saved merged metadata to:", metadata_path)
print("Saved merged topic top-30 words to:", merged_topic_top30_path)
print("Saved consistent topic label table to:", consistent_topic_labels_path)
print("Saved df_combined + Topic export to:", df_combined_topic_path)
print("Saved merge summary to:", merge_summary_path)
print("Saved merged topic overview to:", merge_overview_path)
print("Saved frozen rerun snapshot to:", snapshot_dir)


## 6. Stop here

This notebook is done once the merged model and caches are saved.

Thesis note: if you want the exact finalized thesis run, restore or analyze the frozen snapshot `02_TopicModeling/outputs/frozen_merged_runs/2026-04-03_merged_v1`. The canonical save path may be overwritten by later reruns; the frozen snapshot is the audit-safe reference.

Continue in `Merged_BERTopic_Analysis.ipynb` for:
- readable topic labels
- outlet x topic pivot tables
- topic-label article exports
- balanced and dominance-corrected UMAPs
- BERTopic HTML visuals
- sampling exports